# Practical 1 — Introduction

**Large Language Models in Finance · Chapter 1 / Lecture 1**

You will build the chapter's core text-to-vector tools yourself and run them on real
filings: a TF-IDF representation of earnings-call sentences, a PCA projection of the
vocabulary, scaled dot-product attention, and a small SEC EDGAR data pipeline.

**Setup.** From the repo root, `pip install -e code` installs the `llmfin` helpers used
below (the setup cell also works without installing). Parts 1–3 run offline; Part 4 calls
the live SEC EDGAR API, so set `EDGAR_USER_AGENT` and `RUN_NETWORK = True` to run it.


In [ ]:
# --- setup: make the shared `llmfin` package importable (with or without `pip install -e code`) ---
import sys, pathlib
_here = pathlib.Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / 'code' / 'src' / 'llmfin').is_dir():
        sys.path.insert(0, str(_p / 'code' / 'src')); break

import numpy as np
import matplotlib.pyplot as plt
from llmfin import text, edgar
print('llmfin loaded — helpers:', [n for n in dir(text) if not n.startswith('_')])


## Part 1 — TF-IDF on earnings-call sentences  `[B]`

Compute a TF-IDF representation of the sentences below with `text.tfidf`, then find the
**single most distinctive term** in each sentence (the highest-weighted term).

**Your task:** complete `top_term(i)` so it returns the term with the largest TF-IDF weight
in sentence `i`. Then run the self-check.


In [ ]:
SENTENCES = [
    'Revenue rose sharply on strong iPhone demand',
    'Net interest income climbed as deposits stabilised',
    'The airline cut guidance on rising fuel costs',
    'Cloud revenue accelerated on enterprise AI adoption',
]

terms, M = text.tfidf(SENTENCES)   # terms: list[str], M: (n_docs, n_terms) TF-IDF matrix
print('vocabulary size:', len(terms), '| matrix shape:', M.shape)

def top_term(i):
    # TODO: return terms[j] for the column j with the largest weight in row i of M
    j = int(np.argmax(M[i]))
    return terms[j]

for i, s in enumerate(SENTENCES):
    print(f'{top_term(i):>12}  <-  {s}')


In [ ]:
# self-check (Part 1)
assert M.shape == (len(SENTENCES), len(terms))
# the chosen term must be a maximum-weight (most distinctive) term of its sentence,
# and must not be 'revenue' (which is shared across sentences, so less distinctive)
for i in range(len(SENTENCES)):
    max_terms = {terms[j] for j in np.where(np.isclose(M[i], M[i].max()))[0]}
    assert top_term(i) in max_terms, (i, top_term(i), max_terms)
assert top_term(0) != 'revenue'
print('Part 1 OK')


## Part 2 — Project the vocabulary to 2-D with PCA  `[I]`

Treat each **term** as a vector over documents (the columns of `M`). Reduce those term
vectors to two dimensions with PCA — implemented from scratch via the SVD, no scikit-learn.

**Your task:** complete `pca_2d(X)` so it returns the 2-D PCA scores of the rows of `X`.


In [ ]:
def pca_2d(X):
    X = np.asarray(X, dtype=float)
    Xc = X - X.mean(axis=0, keepdims=True)      # center
    # TODO: SVD and project onto the top 2 right-singular vectors
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    return Xc @ Vt[:2].T

term_vectors = M.T                  # (n_terms, n_docs)
coords = pca_2d(term_vectors)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(coords[:, 0], coords[:, 1], s=12)
for (x, y), t in zip(coords, terms):
    ax.annotate(t, (x, y), fontsize=7)
ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2'); ax.set_title('Vocabulary in 2-D (PCA)')
plt.tight_layout(); plt.show()


In [ ]:
# self-check (Part 2)
assert coords.shape == (len(terms), 2)
assert np.allclose(coords.mean(axis=0), 0, atol=1e-8)   # PCA scores are centered
print('Part 2 OK')


## Part 3 — Scaled dot-product attention from scratch  `[A]`

Implement the attention operation from Chapter 1:

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\tfrac{QK^\top}{\sqrt{d_k}}\right)V$$

**Your task:** complete `softmax` (row-wise, numerically stable) and `attention`.


In [ ]:
def softmax(x):
    # TODO: row-wise softmax; subtract the row max for numerical stability
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)

def attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    W = softmax(scores)
    return W @ V, W

rng = np.random.default_rng(0)
Q, K, V = rng.normal(size=(3, 4)), rng.normal(size=(3, 4)), rng.normal(size=(3, 5))
out, W = attention(Q, K, V)
print('attention output shape:', out.shape)
print('attention weights (rows sum to 1):', W.sum(axis=1).round(3))


In [ ]:
# self-check (Part 3)
assert np.allclose(W.sum(axis=1), 1.0), 'attention rows must sum to 1'
assert out.shape == (3, 5)
# softmax sanity vs a reference
ref = np.exp([1.0, 2.0, 3.0]); ref = ref / ref.sum()
assert np.allclose(softmax(np.array([[1.0, 2.0, 3.0]]))[0], ref)
print('Part 3 OK')


## Part 4 — SEC EDGAR data lab  `[B]` `[I]` `[A]`  *(requires internet)*

Real filings via the free SEC EDGAR REST API through `llmfin.edgar`. Set a descriptive
User-Agent first (SEC fair-access policy), then flip the flag to run:

```bash
export EDGAR_USER_AGENT='Your Name your-email@example.com'
```


In [ ]:
RUN_NETWORK = False   # set True (with EDGAR_USER_AGENT set) to run Part 4


In [ ]:
# [B] Filing browser — most recent filings + the latest 10-K for a ticker
if RUN_NETWORK:
    cik = edgar.get_cik('AAPL')
    f = edgar.get_submissions(cik)['filings']['recent']
    idx, _ = edgar.latest_10k(edgar.get_submissions(cik))
    for k in range(10):
        print(f"{f['form'][k]:>8}  {f['filingDate'][k]}  {f['accessionNumber'][k]}")
    print('Most recent 10-K:', f['filingDate'][idx])
else:
    print('RUN_NETWORK is False — skipping the live EDGAR calls.')


In [ ]:
# [I] MD&A keyword landscape — top distinctive terms in the latest 10-K's MD&A
if RUN_NETWORK:
    mda = edgar.extract_mda(edgar.fetch_10k_html('AAPL'))
    sentences = [s for s in mda.split('. ') if len(s.split()) > 4][:60]
    terms_mda, M_mda = text.tfidf(sentences)
    weight = M_mda.sum(axis=0)
    top = sorted(zip(weight, terms_mda), reverse=True)[:15]
    print('MD&A word count:', len(mda.split()))
    for w, t in top:
        print(f'  {t:<16} {w:.3f}')
else:
    print('RUN_NETWORK is False — skipping.')


In [ ]:
# [A] Cross-sector vocabulary overlap — Jaccard similarity of MD&A vocabularies
if RUN_NETWORK:
    TICKERS = ['AAPL', 'MSFT', 'JPM', 'BAC', 'XOM']
    vocab = {}
    for t in TICKERS:
        try:
            vocab[t] = set(text.tokenize(edgar.extract_mda(edgar.fetch_10k_html(t))))
            print(f'{t}: {len(vocab[t])} unique tokens')
        except Exception as e:
            print(f'{t}: ERROR {e}')
    n = len(TICKERS); J = np.zeros((n, n))
    for i, a in enumerate(TICKERS):
        for k, b in enumerate(TICKERS):
            if a in vocab and b in vocab:
                inter = len(vocab[a] & vocab[b]); union = len(vocab[a] | vocab[b])
                J[i, k] = inter / union if union else 0
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(J, cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(n)); ax.set_xticklabels(TICKERS)
    ax.set_yticks(range(n)); ax.set_yticklabels(TICKERS)
    for i in range(n):
        for k in range(n):
            ax.text(k, i, f'{J[i,k]:.2f}', ha='center', va='center', fontsize=8)
    plt.colorbar(im, ax=ax, label='Jaccard'); plt.tight_layout(); plt.show()
else:
    print('RUN_NETWORK is False — skipping.')


## What you built

- TF-IDF and the most-distinctive-term idea, **in code** (`Part 1`)
- PCA from the SVD, by hand (`Part 2`)
- Scaled dot-product attention from scratch (`Part 3`)
- A reusable SEC EDGAR pipeline via `llmfin.edgar` (`Part 4`)
